# Dunnhumby M5-C 마지막 1주 · 10시드 최종 비교

- 학습: DAY 1~704 전체 / test: DAY 705~711 / validation·holdout 없음
- seed 42~51, 고정 300 epoch, 마지막 체크포인트만 test 1회 평가
- M1 / M3 / 관측 CLV M5-C / degree층 CLV 공동순열 M5-C = 총 40회
- 중단 후 같은 실행 셀을 다시 누르면 저장된 epoch에서 재개합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임(L4 권장)을 연결하세요.'
SOURCE_COMMIT = '58ff14e'
REPO = Path('/content/clv-m5c-final10-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', torch.cuda.get_device_name(0), '| code:', SOURCE_COMMIT)


In [ ]:
import clv_m5_user_centered_final10 as screen
cfg = screen.configure()
print(json.dumps(screen.preflight_summary(cfg), ensure_ascii=False, indent=2))
prepared = screen.prepare(cfg)
stats = prepared['data']['data_stats']
assert set(prepared['data']['splits']) == {'test'}
assert stats['split_rows']['val'] == 0 and stats['split_rows']['holdout'] == 0
assert float(stats['split_boundaries']['train']['end_inclusive']) == 704.0
assert float(stats['split_boundaries']['test']['end_inclusive']) == 711.0
print('사전점검 완료 — 이제 실행 셀에서만 학습이 시작됩니다.')


In [ ]:
result = screen.run(cfg, prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


In [ ]:
from IPython.display import display
metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10']
display(result['summary'][result['summary'].metric.isin(metrics)].sort_values(['metric','model_id']))
display(result['comparison'][result['comparison'].metric.isin(metrics)])


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
output = Path('/content/clv_m5_user_centered_final10_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
    archive.write(prepared['run_dir'] / 'protocol.json', 'protocol.json')
files.download(str(output))
